# Logistic Regression Hyperparameter Tuning

Hyperparameters are settings chosen before fitting. This notebook tunes logistic regression's regularization strength `C` and class weighting with five-fold cross-validation. It uses an imbalanced synthetic dataset so `class_weight` has a practical role; the test set is reserved for the final check.


## 1. Prepare the data

Class `1` is the minority class. Split the data before model selection and use stratification to keep class proportions similar. The grid search will use only the training split.


In [1]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


X, y = make_classification(
    n_samples=1200,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    weights=[0.95, 0.05],
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


Training rows: 960
Test rows: 240


## 2. Choose a search space

`C` is the inverse of regularization strength: smaller values penalize large coefficients more, and larger values penalize them less. Scikit-learn's default is L2 regularization with `C=1`. `class_weight="balanced"` increases the influence of the less frequent class; `None` treats classes equally.

The pipeline standardizes features before fitting. Keeping the scaler inside the pipeline ensures it is refitted separately within each cross-validation fold.


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
parameter_grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10, 100],
    "logisticregression__class_weight": [None, "balanced"],
}


## 3. Run GridSearchCV on training data

`GridSearchCV` tests all 10 combinations using five folds: 50 cross-validation fits, then a refit of the winner on all training data. F1 is used because the positive class is rare. Choose a scoring metric that reflects the task's costs; the test set must not guide the search.


In [3]:
from sklearn.model_selection import GridSearchCV


grid_search = GridSearchCV(
    pipeline,
    parameter_grid,
    cv=5,
    scoring="f1",
)
grid_search.fit(X_train, y_train)

print("Best settings:", grid_search.best_params_)
print("Best mean cross-validation F1:", round(grid_search.best_score_, 3))

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results[
    [
        "param_logisticregression__C",
        "param_logisticregression__class_weight",
        "mean_test_score",
        "std_test_score",
    ]
].sort_values("mean_test_score", ascending=False).round(3)


Best settings: {'logisticregression__C': 10, 'logisticregression__class_weight': None}
Best mean cross-validation F1: 0.362


,param_logisticregression__C,param_logisticregression__class_weight,mean_test_score,std_test_score
6,10.00,NaN,0.362,0.104
8,100.00,NaN,0.362,0.104
4,1.00,NaN,0.309,0.122
1,0.01,balanced,0.242,0.049
3,0.10,balanced,0.239,0.047
5,1.00,balanced,0.237,0.049
9,100.00,balanced,0.237,0.050
7,10.00,balanced,0.237,0.050
2,0.10,NaN,0.106,0.087
0,0.01,NaN,0.000,0.000


## 4. Evaluate the selected model on the test set

The best cross-validation score is for model selection, not final evaluation. After choosing settings, evaluate the refitted model once on the untouched test set. Inspect the confusion matrix and minority-class precision, recall, and F1; a higher score does not guarantee that the model is appropriate for every use.


In [4]:
from sklearn.metrics import classification_report, confusion_matrix


best_model = grid_search.best_estimator_
test_predictions = best_model.predict(X_test)

print("Confusion matrix (rows: actual 0/1, columns: predicted 0/1):")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["majority (0)", "minority (1)"],
        zero_division=0,
    )
)


Confusion matrix (rows: actual 0/1, columns: predicted 0/1):
[[227   0]
 [ 10   3]]
              precision    recall  f1-score   support

majority (0)       0.96      1.00      0.98       227
minority (1)       1.00      0.23      0.38        13

    accuracy                           0.96       240
   macro avg       0.98      0.62      0.68       240
weighted avg       0.96      0.96      0.95       240



## Key ideas

- Smaller `C` means stronger regularization; tune it with cross-validation.
- Search `C` and class weighting using training data only.
- Use the held-out test set once for the final evaluation.
